# Baseline training: fine-tuning ResNet on EuroSAT

This notebook builds up the training pipeline step by step.

In [1]:
from pathlib import Path

from torchvision.datasets import EuroSAT

DATA_ROOT = Path("../data")

## Load the dataset

In [2]:
dataset = EuroSAT(root=DATA_ROOT, download=True)

print(f"Number of images: {len(dataset)}")
print(f"Classes: {dataset.classes}")

Number of images: 27000
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## Split into train / val / test (stratified)

70/15/15, stratified so each split keeps the same class proportions as the full dataset. Done in two steps: first carve out the test set, then split the remainder into train/val. The test set is set aside now and won't be touched again until final evaluation.

In [3]:
from sklearn.model_selection import train_test_split

labels = [label for _, label in dataset.samples]
indices = list(range(len(dataset)))

train_idx, test_idx = train_test_split(
    indices, test_size=0.15, stratify=labels, random_state=42,
)
train_labels = [labels[i] for i in train_idx]
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.15 / 0.85, stratify=train_labels, random_state=42,
)

print(f"Train: {len(train_idx):5d} ({len(train_idx) / len(dataset):.1%})")
print(f"Val:   {len(val_idx):5d} ({len(val_idx) / len(dataset):.1%})")
print(f"Test:  {len(test_idx):5d} ({len(test_idx) / len(dataset):.1%})")

Train: 18899 (70.0%)
Val:    4051 (15.0%)
Test:   4050 (15.0%)


### Sanity check: class balance preserved across splits

In [4]:
import numpy as np

def class_distribution(idx):
    counts = np.bincount([labels[i] for i in idx], minlength=len(dataset.classes))
    return counts / counts.sum()

print(f"{'class':25s} {'train':>8s} {'val':>8s} {'test':>8s}")
for c, class_name in enumerate(dataset.classes):
    train_pct = class_distribution(train_idx)[c]
    val_pct = class_distribution(val_idx)[c]
    test_pct = class_distribution(test_idx)[c]
    print(f"{class_name:25s} {train_pct:8.1%} {val_pct:8.1%} {test_pct:8.1%}")

class                        train      val     test
AnnualCrop                   11.1%    11.1%    11.1%
Forest                       11.1%    11.1%    11.1%
HerbaceousVegetation         11.1%    11.1%    11.1%
Highway                       9.3%     9.3%     9.3%
Industrial                    9.3%     9.3%     9.3%
Pasture                       7.4%     7.4%     7.4%
PermanentCrop                 9.3%     9.3%     9.3%
Residential                  11.1%    11.1%    11.1%
River                         9.3%     9.3%     9.3%
SeaLake                      11.1%    11.1%    11.1%


## Next step

Define transforms (normalization + augmentation on train only) and wrap each split into a `DataLoader`.